In [1]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
from datetime import date

import matplotlib.pyplot as plt 
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter

import eurostat #python wrapper for taking data. 
import time

In [2]:
Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Code\LNG terminals'

In [3]:
import os
os.chdir(Share_point)

In [4]:
os.getcwd()

'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals'

In [5]:
Points_APIs = pd.read_excel(r'Dataproviders 24.01.2023.xlsx',sheet_name='LNG terminals')

### key for version 6 and 7 of the API calls GIE

In [6]:
headers = {"x-key":"d8561648296cb38e6c400823755689941530"} # After July 4th 2022

### Restart here

In [7]:
d1 = '2023-01-01'
d2 = '2023-06-30' # To change in the new year

In [8]:
dates = []
sendOut = []
stored = []
max_storage = []
max_sendout =[]
country = []
Name = []

for item in range(len(Points_APIs)):
        url = Points_APIs.iloc[item,3]+'&from={}&to={}&size=300'.format(d1,d2)
        try:
            r = requests.get(url,headers=headers)
            if r.status_code != 200:
                print(r.status_code)
                print(url)

            raw_data = r.json()
            inner_data = raw_data['data']
            for x in inner_data:
                #here we work with the APIs
                date = x['gasDayStart']
                inventory = x['inventory']
                sendOuts = x['sendOut']
                Max_Storage = x['dtmi']
                Max_Sendout = x['dtrs']
                #here we work with the excel file
                countries = Points_APIs.iloc[item,0]
                Names = Points_APIs.iloc[item,5]


                dates.append(date)     
                sendOut.append(sendOuts)
                stored.append(inventory)
                max_storage.append(Max_Storage)
                max_sendout.append(Max_Sendout)
                country.append(countries)
                Name.append(Names)
        except Exception as e:
            print(e)

In [9]:
df = pd.DataFrame()
df['dates'] = dates
df['sendOut'] = sendOut          ## in GWh/d
df['stored'] = stored            ## in 10^3 m^3 LNG
df['Max Storage'] = max_storage  ## in 10^3 m^3 LNG
df['Max Sendout'] = max_sendout  ## in GWh/d
df['country'] = country
df['name'] = Name

In [10]:
# Getting rid of NaNa
df.replace('-', np.NaN,inplace=True)
# Getting rid of observations for which we have no data
df = df.dropna()

In [11]:
df['sendOut'] = pd.to_numeric(df['sendOut'])
df['stored'] = pd.to_numeric(df['stored'])
df['Max Storage'] = pd.to_numeric(df['Max Storage']) 
df['Max Sendout'] = pd.to_numeric(df['Max Sendout']) 

In [12]:
df.head()

,dates,sendOut,stored,Max Storage,Max Sendout,country,name
0,2023-03-27,394.8,431.49,566.0,541.0,BE,Zeebrugge LNG Terminal
1,2023-03-26,487.9,420.43,566.0,541.0,BE,Zeebrugge LNG Terminal
2,2023-03-25,477.8,178.74,566.0,518.4,BE,Zeebrugge LNG Terminal
3,2023-03-24,466.1,250.02,566.0,541.0,BE,Zeebrugge LNG Terminal
4,2023-03-23,447.7,522.37,566.0,541.0,BE,Zeebrugge LNG Terminal


In [13]:
from datetime import date
df.to_csv('raw_data/2023_1.csv')

###  merge with historic data

In [14]:
# Change current directory
import os
os.chdir(Share_point + '\\raw_data')

In [15]:
df_5a = pd.read_csv('2019_1.csv',index_col=0)
df_5b = pd.read_csv('2019_2.csv',index_col=0)
df_6a = pd.read_csv('2020_1.csv',index_col=0)
df_6b = pd.read_csv('2020_2.csv',index_col=0)
df_7a = pd.read_csv('2021_1.csv',index_col=0)
df_7b = pd.read_csv('2021_2.csv',index_col=0)
df_8a = pd.read_csv('2022_1.csv',index_col=0)
df_8b = pd.read_csv('2022_2.csv',index_col=0)
df_9a = pd.read_csv('2023_1.csv',index_col=0)

In [16]:
df = pd.concat([df_5a,df_5b,df_6a,df_6b,df_7a,df_7b,df_8a,df_8b,df_9a])

In [17]:
# del df['Unnamed: 0']
df = df.drop_duplicates()

In [18]:
df.tail()

,dates,sendOut,stored,Max Storage,Max Sendout,country,name
2145,2023-01-05,616.7,2165.24,3316.5,1910.4,ES*,TVB (Virtual balancing LNG tank)
2146,2023-01-04,550.4,2259.05,3316.5,1910.4,ES*,TVB (Virtual balancing LNG tank)
2147,2023-01-03,612.7,2206.34,3316.5,1910.4,ES*,TVB (Virtual balancing LNG tank)
2148,2023-01-02,536.2,2303.10,3316.5,1910.4,ES*,TVB (Virtual balancing LNG tank)
2149,2023-01-01,338.4,2422.49,3316.5,1910.4,ES*,TVB (Virtual balancing LNG tank)


In [19]:
df = df.set_index(pd.DatetimeIndex(df['dates']))

In [20]:
del df['dates']

In [21]:
df = df.sort_values(by='dates')

In [22]:
df

,sendOut,stored,Max Storage,Max Sendout,country,name
dates,,,,,,
2019-01-01,114.9,776.21,1026.40,644.0,GB,Isle of Grain LNG Terminal
2019-01-01,68.4,258.85,600.00,519.8,FR,Dunkerque LNG Terminal
2019-01-01,26.0,440.71,585.48,375.8,ES,Cartagena LNG Terminal
2019-01-01,228.2,178.64,250.00,228.5,IT,Rovigo LNG Terminal
2019-01-01,86.5,197.80,225.00,205.5,GR,Revythoussa LNG Terminal
...,...,...,...,...,...,...
2023-03-27,271.7,59.63,250.00,274.3,IT,Rovigo LNG Terminal
2023-03-27,40.1,12.69,137.16,166.5,IT,FSRU OLT Offshore LNG Toscana
2023-03-27,902.5,1731.81,3316.50,1910.4,ES*,TVB (Virtual balancing LNG tank)


In [23]:
# Change current directory
import os
os.chdir(Share_point)

In [24]:
df_C = df.groupby(['dates','country']).sum().reset_index()

In [25]:
df_C = df_C.set_index(pd.DatetimeIndex(df_C['dates']))

In [26]:
from datetime import date
today = date.today()
last_obs = today - timedelta(days=2)
print(last_obs)
# enddate = datetime.strptime('2022-07-17', "%d-%m-%Y")
enddate = last_obs
startdate = enddate - timedelta(days=30)

2023-03-27


### Weekly data

In [27]:
df_w = df_C.groupby([pd.Grouper(freq='W'),'country']).sum().reset_index()

In [28]:
df_w=df_w.set_index(pd.DatetimeIndex(df_w['dates']))

In [29]:
df_w.country.unique()

array(['BE', 'ES', 'FR', 'GB', 'GR', 'IT', 'LT', 'NL', 'PL', 'PT', 'ES*',
       'GB*', 'HR', 'DE'], dtype=object)

In [30]:
df_w = df_w[~df_w.country.isin(['ES*','GB','GB*'])] # Get rid of UK and ES TVB aggregate for which there are no data pre-2020

In [31]:
del df_w['dates']

In [32]:
## Can change from new year
data = pd.DataFrame()
data['week'] = list(range(0,54,1))

# plot = df_w.groupby([pd.Grouper(freq='W'),'country']).sum()[:] ## use a converter if you want to transform the data
df_w['week'] = df_w.index.isocalendar().week

hist = df_w['2019-01-05':'2022-01-02'].groupby(['country','week']).mean().reset_index()
values2021 = df_w.loc['2019-12-28':'2022-01-02'].groupby(['country','week']).max().reset_index()
values2022 = df_w.loc['2022-01-03':'2023-01-01'].groupby(['country','week']).max().reset_index()
values2023 = df_w.loc['2023-01-01':'2023-12-31'].groupby(['country','week']).max().reset_index()

In [33]:
hist['regas %'] = (hist['sendOut']/hist['Max Sendout'])*100
values2021['regas %'] = (values2021['sendOut']/values2021['Max Sendout'])*100
values2022['regas %'] = (values2022['sendOut']/values2022['Max Sendout'])*100
values2023['regas %'] = (values2023['sendOut']/values2023['Max Sendout'])*100

In [34]:
import os
os.chdir(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data\LNG')

In [35]:
Excelwriter = pd.ExcelWriter(r"GIE ALSI LNG terminals\LNG_GIE_weekly_data {}.xlsx".format(today),engine="xlsxwriter")
hist.to_excel(Excelwriter, sheet_name="2019-2021", index=False)
values2021.to_excel(Excelwriter, sheet_name="2021", index=False)
values2022.to_excel(Excelwriter, sheet_name="2022", index=False)
values2023.to_excel(Excelwriter, sheet_name="2023", index=False)
Excelwriter.close()
Excelwriter.save()

C:\Users\giovanni.sgaravatti\Anaconda3\lib\site-packages\xlsxwriter\workbook.py:339: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
